# Midterm Ai
# Camilo Vásquez


## 1. Bookstores


In [1]:
from pathlib import Path  
import numpy as np  
import pandas as pd  
from sklearn.model_selection import train_test_split  
from sklearn.impute import SimpleImputer  
from sklearn.preprocessing import RobustScaler, OneHotEncoder  
from sklearn.compose import ColumnTransformer  
from sklearn.pipeline import Pipeline  
from sklearn.neural_network import MLPClassifier 
from sklearn.metrics import accuracy_score, roc_auc_score 

# 2. Data procesiing


In [3]:
DATA_DIR = Path.cwd()  
if not (DATA_DIR / "X_train_Kaggle.csv").exists():  
    DATA_DIR = DATA_DIR / "Mid AI"  
if not (DATA_DIR / "X_train_Kaggle.csv").exists():  
    DATA_DIR = Path(r"C:\Users\THINKPAD\Documents\GitHub\Ai-Yachay-2026\Mid AI")
    
X_raw = pd.read_csv(DATA_DIR / "X_train_Kaggle.csv", low_memory=False) 
X_test_raw = pd.read_csv(DATA_DIR / "X_test_Kaggle.csv", low_memory=False)  
y_raw = pd.read_csv(DATA_DIR / "y_train_Kaggle.csv")  
sample = pd.read_csv(DATA_DIR / "sample_submission_credit.csv")  
class_map = {"Good": 0, "Poor": 1, "Standard": 2}  
y_text = y_raw["Credit_Score"].astype(str).str.strip()  
y = y_text.map(class_map)  

if y.isna().any():  
    raise ValueError("Hay una clase que no aparece en class_map")  

y = y.astype(int)  

## 3. cleaning

In [4]:
drop_cols = ["ID", "id", "Customer_ID", "Name", "SSN", "Month", "Type_of_Loan", "Unnamed: 0"] 
numeric_cols = [  
    "Age", "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts",
    "Num_Credit_Card", "Interest_Rate", "Num_of_Loan", "Delay_from_due_date",
    "Num_of_Delayed_Payment", "Changed_Credit_Limit", "Num_Credit_Inquiries",
    "Outstanding_Debt", "Credit_Utilization_Ratio", "Total_EMI_per_month",
    "Amount_invested_monthly", "Monthly_Balance"
]
category_cols = ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]  
def clean_features(data):  
    data = data.drop(columns=drop_cols, errors="ignore").copy()  
    if "Credit_History_Age" in data.columns:  
        age_text = data["Credit_History_Age"].astype("string")  
        age_parts = age_text.str.extract(r"(\d+)\s*Years?.*?(\d+)\s*Months?")  
        years = pd.to_numeric(age_parts[0], errors="coerce")  
        months = pd.to_numeric(age_parts[1], errors="coerce")  
        data["Credit_History_Months"] = (years * 12 + months).astype(float)  
        data = data.drop(columns="Credit_History_Age")  

    for column in numeric_cols:  
        if column in data.columns:  
            text = data[column].astype("string").str.replace("_", "", regex=False).str.strip()  
            data[column] = pd.to_numeric(text, errors="coerce").astype(float)  
            data[column] = data[column].replace([np.inf, -np.inf], np.nan)  

    missing_tokens = ["", "nan", "None", "NM", "!@9#%8"]  
    for column in category_cols:  
        if column in data.columns:  
            text = data[column].astype(str).str.strip() 
            text = text.str.replace(r"^_+$", "", regex=True) 
            data[column] = text.replace(missing_tokens, np.nan) 

    return data  

X = clean_features(X_raw)  
X_test = clean_features(X_test_raw).reindex(columns=X.columns)  
numeric_features = X.select_dtypes(include="number").columns.tolist()  
categorical_features = X.select_dtypes(exclude="number").columns.tolist()  

## 4. Train




In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(  
    X, y, test_size=0.2, random_state=42, stratify=y  
)

def clip_numeric(train_data, other_data, columns):  
    train_data = train_data.copy()  
    other_data = other_data.copy()  
    for column in columns:  
        lower = train_data[column].quantile(0.01)  
        upper = train_data[column].quantile(0.99) 
        train_data[column] = train_data[column].clip(lower, upper)  
        other_data[column] = other_data[column].clip(lower, upper) 
    return train_data, other_data  

X_train, X_valid = clip_numeric(X_train, X_valid, numeric_features) 

##  MLP


In [ ]:
numeric_prep = Pipeline(steps=[ 
    ("imputer", SimpleImputer(strategy="median")),  
    ("scaler", RobustScaler())  
])
category_prep = Pipeline(steps=[  
    ("imputer", SimpleImputer(strategy="most_frequent")),  
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))  
])
preprocessor = ColumnTransformer(transformers=[  
    ("numbers", numeric_prep, numeric_features),  
    ("categories", category_prep, categorical_features)  
])
model = Pipeline(steps=[  
    ("prep", preprocessor),  
    ("mlp", MLPClassifier(  
        hidden_layer_sizes=(64, 32), activation="relu", solver="adam",  
        alpha=0.01, early_stopping=True, max_iter=300, random_state=42  
    ))
])
model = model.fit(X_train, y_train)  

## 6. Measure results




In [7]:
valid_predictions = model.predict(X_valid)  
valid_probabilities = model.predict_proba(X_valid)  

print("Accuracy de validación:", round(accuracy_score(y_valid, valid_predictions), 4))  
print("ROC-AUC macro OVR:", round(  
    roc_auc_score(y_valid, valid_probabilities, multi_class="ovr", average="macro", labels=[0, 1, 2]), 4  
))

Accuracy de validación: 0.6794
ROC-AUC macro OVR: 0.8343


## 7. Train with all data

In [10]:
X_full, X_test_ready = clip_numeric(X, X_test, numeric_features)  
model = model.fit(X_full, y)  
test_predictions = model.predict(X_test_ready).astype(int)  

if len(sample) != len(test_predictions): 
    raise ValueError("La plantilla y X_test tienen cantidades de filas distintas")  
if "class" not in sample.columns:  
    raise ValueError("La plantilla no contiene la columna class")  

submission = sample.copy()  
submission["class"] = np.array(["Good", "Poor", "Standard"])[test_predictions]  
output_file = DATA_DIR / "submission.csv"  
submission.to_csv(output_file, index=False)  

print("Entrega guardada en:", output_file)  

Entrega guardada en: c:\Users\THINKPAD\Documents\GitHub\Ai-Yachay-2026\Mid AI\submission.csv
